In [ ]:
%reload_ext autoreload
%autoreload 2
import sys
import os

# Adjust the path to your project root if needed
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
    
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import statsmodels.api as sm
from functions.functions_comp import DataAggregatorEvaluator
from matplotlib.lines import Line2D

merged_df = pd.read_csv('../data/merged_data.csv')

# Using aggregation to compare gas consumption data to syntetic heat energy used

In [ ]:
group_size = 1

merged_df.dropna(subset=['qnutzwaerme_2020_kwh'], inplace=True)
merged_df.reset_index(drop=True, inplace=True)
DAE = DataAggregatorEvaluator(merged_df)
aggregated_df = DAE.aggregate_data(method='random', n_groups=group_size)
aggregated_df.dropna(subset=['qnutzwaerme_2020_kwh'], inplace=True)
print(f"Number of rows after dropping NaN values: {len(aggregated_df)}")
mse = mean_squared_error(aggregated_df['2020'], aggregated_df['qnutzwaerme_2020_kwh'])
rmse = np.sqrt(mse)
mape = np.mean(np.abs((aggregated_df['2020'] - aggregated_df['qnutzwaerme_2020_kwh']) / aggregated_df['2020'])) * 100
rmse = np.sqrt(mse)

# Calculate the mean of the actual values (assuming '2020' column holds the actual values)
mean_actual = np.mean(aggregated_df['2020'])

# Calculate the percentage of RMSE on the mean
percentage_rmse = (rmse / mean_actual) * 100
print(f"Root Mean Squared Error (RMSE): {percentage_rmse:.2f}%")
print(f"Mean Absolute Percentage Error (MAPE): {mape:.2f}%")
aggregated_df['residuals'] = aggregated_df['2020'] - aggregated_df['qnutzwaerme_2020_kwh']
plt.figure(figsize=(12, 8))

plt.scatter(aggregated_df['2020'], aggregated_df['qnutzwaerme_2020_kwh'], label='Data Points', color='blue')

max_val = max(merged_df['2020'].max(), aggregated_df['qnutzwaerme_2020_kwh'].max())
min_val = min(aggregated_df['2020'].min(), 0)
plt.plot([min_val, max_val], [min_val, max_val], color='green', linestyle='--', label='y = x Line')

for _, row in aggregated_df.iterrows():
    plt.plot([row['2020'],row['2020']], [row['2020'],row['qnutzwaerme_2020_kwh']], color='red', linestyle='-', linewidth=1)
handles, labels = plt.gca().get_legend_handles_labels()
handles.append(Line2D([0], [0], color='red', linestyle='-', linewidth=1))
labels.append('Residuals')
plt.xlabel('2020 Real Data (kWh)')
plt.ylabel('2020 Synthetic Data (kWh)')
plt.title('Real vs Synthetic Energy Consumption Data')
plt.tight_layout()
plt.legend(handles=handles, labels=labels)
plt.grid(True)
plt.show()

In [ ]:
distance_thresholds = range(0, 500, 1) 
results = []
agg_data = []

for threshold in distance_thresholds:
    agg_data.append(DAE.aggregate_data(method='distance', distance_threshold=threshold))
    metrics = DAE.aggregate_and_evaluate(threshold)
    metrics['distance_threshold'] = threshold
    results.append(metrics)

results_df = pd.DataFrame(results)

In [ ]:
DAE.plot_metric(results_df, metric='rmse_perc', evaluation_type='distance_threshold')

In [ ]:
DAE.plot_metric(results_df, metric='rmse_perc', evaluation_type='distance_threshold')

In [ ]:
n_groups = range(1, 51, 1) 
results = []
results_std = []

for n in n_groups:
    metrics_mean, metrics_std = DAE.aggregate_and_evaluate_random(n, 100)
    metrics_mean['group_size'] = n
    results.append(metrics_mean)
    results_std.append(metrics_std)
results_df = pd.DataFrame(results)
results_std = pd.DataFrame(results_std)

In [ ]:
DAE.plot_metric(results_df, metric='rmse', evaluation_type='distance_threshold')

In [ ]:
DAE.plot_metric(results_df, results_std, metric='rmse_perc', evaluation_type='group_size')